# Solving Ground States

This notebook demonstrates how to solve for ground states of quantum many-body systems using both exact diagonalization and DMRG.

## Topics Covered
- Exact diagonalization (ED) solver
- Density matrix renormalization group (DMRG) solver
- Ground state analysis
- Energy and entanglement calculations
- Comparison between methods


In [1]:
# Add the parent directory to Python path for development
import sys
import os
sys.path.insert(0, os.path.abspath('..'))


### Step 1: Build a Quantum Hamiltonian

Let's start by creating a simple Heisenberg chain Hamiltonian.


In [2]:
from shadowgpt.physics import ham_tf_ising
ham = ham_tf_ising(n=6, gs=[0.43], bc="open")
ham

-0.43 X5 - 0.57 Z4 Z5 - 0.43 X4 - 0.57 Z3 Z4 - 0.43 X3 - 0.57 Z2 Z3 - 0.43 X2 - 0.57 Z1 Z2 - 0.43 X1 - 0.57 Z0 Z1 - 0.43 X0

### Step 2: Exact Diagonalization (ED)

Exact diagonalization is a method to find the ground state by diagonalizing the full Hamiltonian matrix. It's exact but limited to small systems due to exponential scaling with system size.

**When to use ED:**
- Small systems (typically ≤ 20 qubits)
- When you need exact results
- For benchmarking other methods

**Key features:**
- Returns the exact ground state vector
- Computationally expensive for large systems
- Memory scales as 2^n where n is the number of qubits


In [3]:
import numpy as np
from shadowgpt.physics import EDSolver

# Create ED solver with default configuration
ed_solver = EDSolver()
# Solve for ground state
energy, state = ed_solver.solve(ham)
print(f"Ground energy: {energy:.6f}")
print(f"Ground state norm: {np.linalg.norm(state):.6f}")


Ground energy: -3.572820
Ground state norm: 1.000000


### Step 3: Density Matrix Renormalization Group (DMRG)

DMRG is a variational method that finds ground states by optimizing a Matrix Product State (MPS) representation. It's much more efficient than ED for 1D systems and can handle larger system sizes.

**When to use DMRG:**
- 1D quantum systems
- Larger systems (hundreds of sites possible)
- When you need good approximations to ground states
- Systems with limited entanglement

**Key features:**
- Returns a Matrix Product State (MPS) representation
- Scalable to much larger systems than ED
- Configurable bond dimensions and convergence criteria
- Shows convergence information and energy history


In [4]:
from shadowgpt.physics import DMRGSolver, DMRGConfig

# Create DMRG solver with custom configuration
dmrg_config = DMRGConfig(
    bsz=2,
    which='SA',
    tol=0.0001,
    verbosity=1
)
dmrg_solver = DMRGSolver(dmrg_config)

# Solve for ground state
energy, state = dmrg_solver.solve(ham)
print(f"Ground energy: {energy:.6f}")
print(f"Number of sweeps: {len(dmrg_solver.energies)}")
print(f"Converged: {dmrg_solver.converged}")
print(f"Energy history: {dmrg_solver.energies}")

# Show the MPS structure (bond dimensions are displayed here)
print("\nMPS structure:")
state.show()

1, R, max_bond=(10/10), cutoff:1e-09


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/cotengra/hyperoptimizers/hyper.py:55: UserWarning: Couldn't find `optuna`, `cmaes`, or `nevergrad` so will use completely random sampling in place of hyper-optimization. It is recommended to install one of these libraries for higher quality contraction paths.
  warnings.warn(
100%|############################################| 5/5 [00:00<00:00, 250.48it/s]

Energy: (-3.5728199495803565-1.3846049788756385e-15j) ... not converged.
2, R, max_bond=(6/20), cutoff:1e-09



100%|############################################| 5/5 [00:00<00:00, 323.37it/s]

Energy: (-3.572819971603587-1.4431536515108484e-15j) ... converged!
Ground energy: -3.572820-0.000000j
Number of sweeps: 2
Converged: True
Energy history: [(-3.5728199495803565-1.3846049788756385e-15j), (-3.572819971603587-1.4431536515108484e-15j)]

MPS structure:
 2 4 6 4 2 
>─>─>─>─>─●
│ │ │ │ │ │
